# Explore the recommender

Hands-on probes of the trained ALS pipeline. Run cells top-to-bottom; each section is independent after the setup cell.

## Setup

In [17]:
import os, pathlib
# Make relative paths in spotify_recs (e.g. models/als.pkl) resolve from repo root.
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(_root)
print(f'cwd: {_root}')

import numpy as np
import pandas as pd
from spotify_recs.recommender import (
    load_pipeline, recommend_for_history, ARTIST_LOOKUP_PATH, JUNK_NORM_NAMES,
)

pipe = load_pipeline()
lookup = pd.read_parquet(ARTIST_LOOKUP_PATH)  # artist_id, canonical_name, ...
name_to_id = dict(zip(lookup['canonical_name'].str.lower(), lookup['artist_id']))
id_to_name = dict(zip(lookup['artist_id'], lookup['canonical_name']))

# Pull the item-factor matrix out of the trained scorer.
# LensKit 2026.1: scorer node lives in pipe.node('scorer'); .component is ImplicitMFScorer.
scorer = pipe.node('scorer').component
item_factors = np.asarray(scorer.item_embeddings)  # (n_items, d)
item_ids = np.asarray(scorer.items.ids())          # parallel array of artist_id
id_to_row = {aid: i for i, aid in enumerate(item_ids)}
print(f'item_factors: {item_factors.shape}, vocab size: {len(item_ids)}')

cwd: /Users/ryanquinlan/New_Soundz
item_factors: (267739, 64), vocab size: 267739


## 1. Sanity-check the latent space — nearest neighbors

If neighborhoods look right, the model is healthy. If they're full of Smash Mouth / Katy Perry, popularity bias is leaking through cosine.

In [18]:
def neighbors(artist_name: str, k: int = 20) -> pd.DataFrame:
    aid = name_to_id.get(artist_name.lower())
    if aid is None or aid not in id_to_row:
        raise ValueError(f'{artist_name!r} not in CF vocab')
    v = item_factors[id_to_row[aid]]
    norms = np.linalg.norm(item_factors, axis=1)
    sims = (item_factors @ v) / (norms * np.linalg.norm(v) + 1e-12)
    top = np.argsort(-sims)
    rows = []
    for idx in top:
        nm = id_to_name.get(int(item_ids[idx]), '?')
        if nm.lower() in JUNK_NORM_NAMES or int(item_ids[idx]) == aid:
            continue
        rows.append((nm, float(sims[idx])))
        if len(rows) == k: break
    return pd.DataFrame(rows, columns=['neighbor', 'cosine'])

for a in ['Radiohead', 'Daft Punk', 'Kanye West', 'Aphex Twin', 'The Strokes']:
    try:
        print(f'\n=== {a} ==='); print(neighbors(a, 15).to_string(index=False))
    except ValueError as e: print(e)


=== Radiohead ===
           neighbor   cosine
        the beatles 0.619780
          sigur rós 0.556483
              björk 0.488617
               beck 0.476022
           the cure 0.437259
         portishead 0.430679
        arcade fire 0.412124
   atmosphere italy 0.406889
         pink floyd 0.391658
             beirut 0.385023
           ganglion 0.384573
death cab for cutie 0.384247
  animal collective 0.382163
    my toys like me 0.376825
         the please 0.376401

=== Daft Punk ===
             neighbor   cosine
              justice 0.721810
the chemical brothers 0.596421
           digitalism 0.568887
             gorillaz 0.535121
          fatboy slim 0.533897
      katamari damacy 0.495912
        mark imperial 0.484206
           black adam 0.484206
        cosmic nature 0.463828
            nine+ma15 0.463828
            maki+kana 0.463828
                 mylo 0.461284
     keiichi sugiyama 0.460871
  simian mobile disco 0.458134
        basement jaxx 0.455448

=

## 2. Fold-in seed sweep — short_term vs medium_term vs blended

How sensitive is the user vector to the recency weighting? Stub in your own seed lists below.

In [19]:
# Replace these with artists you actually listen to that exist in CF vocab.
short_term = ['Radiohead', 'Aphex Twin', 'Boards of Canada']
medium_term = ['Daft Punk', 'The Strokes', 'LCD Soundsystem', 'Phoenix']

def to_weights(names, weight=1.0):
    out = {}
    for n in names:
        aid = name_to_id.get(n.lower())
        if aid is not None: out[int(aid)] = weight
    return out

scenarios = {
    'short_only': to_weights(short_term, 1.0),
    'medium_only': to_weights(medium_term, 1.0),
    'blend_2x_short': {**to_weights(medium_term, 1.0), **to_weights(short_term, 2.0)},
    'blend_5x_short': {**to_weights(medium_term, 1.0), **to_weights(short_term, 5.0)},
}

for label, w in scenarios.items():
    print(f'\n=== {label} (n_seeds={len(w)}) ===')
    print(recommend_for_history(pipe, w, n=10).to_string(index=False))


=== short_only (n_seeds=3) ===
 item_id canonical_name    score
      34          björk 0.522919
      23      sigur rós 0.483058
      56            air 0.464543
      35      daft punk 0.432333
      74     portishead 0.428045
      63 massive attack 0.415880
     548   squarepusher 0.406611
     437       autechre 0.389876
     280     amon tobin 0.374488
     406      brian eno 0.368754

=== medium_only (n_seeds=4) ===
 item_id        canonical_name    score
     194               justice 0.613093
      86       franz ferdinand 0.528493
      38            bloc party 0.519129
     299              hot chip 0.498726
      97              gorillaz 0.486751
     210                  mgmt 0.483492
      22        arctic monkeys 0.470887
     137 the chemical brothers 0.469171
      69              interpol 0.468891
    1002  peter bjorn and john 0.464735

=== blend_2x_short (n_seeds=7) ===
 item_id        canonical_name    score
     194               justice 0.689406
      56        

## 3. Proxy substitution decay sweep

For an artist NOT in CF vocab (Tyler / JPEGMAFIA / Frank Ocean), pull Last.fm similars, intersect with vocab, fold in with varying decay. The 'best' decay is the one whose recs feel closest to a direct CF query on a stylistically-adjacent matched artist.

In [20]:
from spotify_recs.cache import ArtistCache

# DIAGNOSTIC: weights are ignored by LensKit fold-in (verified in source).
# So the only knob we have is the SET of seeds. Test that the set matters.

TARGET = 'Tyler, The Creator'
DIRECT_SEEDS = ['Kanye West', 'Madvillain', 'Outkast']

with ArtistCache() as cache:
    similars = cache.get_lastfm_similar(TARGET, limit=50)

direct_ids = [int(name_to_id[n.lower()]) for n in DIRECT_SEEDS if n.lower() in name_to_id]
in_vocab = []
for _, canonical, score in similars:
    aid = name_to_id.get(canonical.lower())
    if aid is not None and aid in id_to_row and int(aid) not in direct_ids:
        in_vocab.append((canonical, int(aid), float(score)))
proxy_ids = [aid for _, aid, _ in in_vocab]
print(f'direct: {len(direct_ids)}, proxies in vocab: {len(proxy_ids)}')

# 1) Vary the SET (the only knob LensKit honors with use_ratings=False).
scenarios = {
    'direct_only            (3)': direct_ids,
    'direct + top1_proxy    (4)': direct_ids + proxy_ids[:1],
    'direct + top3_proxies  (6)': direct_ids + proxy_ids[:3],
    'direct + all_proxies  (15)': direct_ids + proxy_ids,
    'proxies_only          (12)': proxy_ids,
}
top10s = {}
for label, ids in scenarios.items():
    weights = {aid: 1.0 for aid in ids}  # weights are ignored anyway, so 1.0 each
    df = recommend_for_history(pipe, weights, n=10)
    top10s[label] = list(df['item_id'])
    print(f'\n=== {label} ===')
    print(df.to_string(index=False))

# 2) Overlap matrix: how many items shared between any two scenarios.
print('\n--- top-10 overlap matrix (rows vs cols) ---')
labels = list(top10s)
print('       ' + '  '.join(f'{i+1}' for i in range(len(labels))))
for i, a in enumerate(labels):
    cells = [f'{len(set(top10s[a]) & set(top10s[b])):2d}' for b in labels]
    print(f'{i+1}: {a:28s} ' + '  '.join(cells))

# 3) Try the OFFICIAL way to pass weights: rating field + use_ratings=True.
#    This requires monkey-patching the scorer config since training was use_ratings=False.
print('\n--- testing use_ratings=True with rating field ---')
from lenskit.data import ItemList
from lenskit.data import RecQuery
from lenskit import recommend

orig_use = scorer.config.use_ratings
try:
    scorer.config.use_ratings = True
    for w_proxy in [0.0, 1.0, 5.0]:
        ids = direct_ids + proxy_ids
        ratings = [1.0]*len(direct_ids) + [w_proxy]*len(proxy_ids)
        hist = ItemList(item_ids=ids, rating=ratings, ordered=False)
        out = recommend(pipe, RecQuery(history_items=hist), n=10)
        names = [id_to_name.get(int(i), '?') for i in out.ids()][:10]
        print(f'\n  proxy_rating={w_proxy}: {names}')
finally:
    scorer.config.use_ratings = orig_use

direct: 3, proxies in vocab: 12

=== direct_only            (3) ===
 item_id       canonical_name    score
     173                jay-z 0.555534
     306            the roots 0.492309
     320          lupe fiasco 0.480343
     449       gnarls barkley 0.475211
     166                  nas 0.455238
     367               common 0.452328
     124            lil wayne 0.425423
     707              mos def 0.413916
     387 a tribe called quest 0.406524
     796              n*e*r*d 0.396025

=== direct + top1_proxy    (4) ===
 item_id       canonical_name    score
     173                jay-z 0.556545
     306            the roots 0.491976
     320          lupe fiasco 0.483660
     449       gnarls barkley 0.475249
     166                  nas 0.456660
     367               common 0.453773
     124            lil wayne 0.429165
     707              mos def 0.415417
     387 a tribe called quest 0.407457
     796              n*e*r*d 0.399707

=== direct + top3_proxies  (6) ===
 i

## 5. Tag distribution audit

Pull `getTopTags` for ~20 artists across genres. Check raw vs denylist-filtered output. Decide whether the 125-genre HF allowlist is too narrow.

In [21]:
from spotify_recs.lastfm_api import TAG_DENYLIST
from collections import Counter

TEST = [
    'Tyler, The Creator', 'Kendrick Lamar', 'Frank Ocean', 'JPEGMAFIA',
    'Radiohead', 'Aphex Twin', 'Boards of Canada', 'Burial',
    'Daft Punk', 'LCD Soundsystem', 'Disclosure', 'Jamie xx',
    'The Strokes', 'Phoenix', 'HYUKOH', 'Malcolm Todd',
    'Anderson .Paak', 'Tame Impala', 'King Krule', 'Mac DeMarco',
]

all_tags_raw, all_tags_clean, per_artist = Counter(), Counter(), {}
with ArtistCache() as cache:
    for a in TEST:
        try:
            tags = cache.get_lastfm_tags(a)  # [(tag, count), ...]
        except Exception as e: print(f'{a}: {e}'); continue
        raw = [t for t, _ in tags]
        clean = [t for t, _ in tags if t.lower() not in TAG_DENYLIST]
        all_tags_raw.update(raw); all_tags_clean.update(clean)
        per_artist[a] = (len(raw), len(clean), clean[:6])

print('per-artist (raw_n, clean_n, top_clean):')
for a, v in per_artist.items(): print(f'  {a:25s} {v[0]:3d} -> {v[1]:3d}  {v[2]}')
print('\nmost common dropped-by-denylist tags:')
dropped = all_tags_raw - all_tags_clean
print(dropped.most_common(15))

per-artist (raw_n, clean_n, top_clean):
  Tyler, The Creator          9 ->   9  ['hip-hop', 'rap', 'ofwgkta', 'underground hip-hop', 'swag', 'hip hop']
  Kendrick Lamar             10 ->  10  ['hip-hop', 'rap', 'west coast', 'hip hop', 'compton', 'california']
  Frank Ocean                 9 ->   9  ['rnb', 'soul', 'hip-hop', 'r&b', 'ofwgkta', 'neo-soul']
  JPEGMAFIA                  10 ->  10  ['hip-hop', 'glitch hop', 'experimental', 'experimental hip hop', 'experimental hip-hop', 'industrial hip hop']
  Radiohead                   9 ->   9  ['rock', 'alternative', 'alternative rock', 'indie', 'electronic', 'experimental']
  Aphex Twin                  9 ->   9  ['electronic', 'idm', 'ambient', 'experimental', 'electronica', 'techno']
  Boards of Canada           10 ->  10  ['ambient', 'electronic', 'idm', 'chillout', 'electronica', 'downtempo']
  Burial                      9 ->   9  ['dubstep', 'ambient', 'electronic', 'experimental', 'electronica', 'future garage']
  Daft Punk    

## What to look for

- §1 — Neighbors should be stylistic peers, not popularity blobs. If Radiohead's neighbors include Smash Mouth, you have a normalization or junk-filter gap.
- §2 — Wildly different recs across short/medium suggests fold-in is dominated by recency. Tame ratios (2x, 5x) help you pick a default for the live path.
- §3 — As decay increases past ~0.5, recs should converge toward what you'd get from a direct CF query on the strongest single proxy. If they diverge instead, the proxy graph is noisy.
- §5 — If the clean tag count drops below ~5 for many artists, the denylist or HF allowlist is too aggressive.